## Why LoRA

Full fine-tuning updates all model weights. LoRA updates only low-rank adapters:

$$W' = W + \Delta W = W + B A$$

where rank $r$ is small relative to full dimension.

## Step 1: Build a LoRA Linear Layer


In [ ]:
#| eval: false
import torch
import torch.nn as nn
import torch.nn.functional as F

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=8, alpha=16, dropout=0.0):
        super().__init__()
        self.rank = rank
        self.scaling = alpha / rank

        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02, requires_grad=False)
        self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=False)

        self.A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        base = F.linear(x, self.weight, self.bias)
        lora = F.linear(self.dropout(x), self.A)
        lora = F.linear(lora, self.B)
        return base + self.scaling * lora


## Step 2: Use LoRA in an MLP Classifier


In [ ]:
#| eval: false
class LoRAClassifier(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, num_classes=3, rank=8):
        super().__init__()
        self.fc1 = LoRALinear(input_dim, hidden_dim, rank=rank, alpha=16)
        self.act = nn.ReLU()
        self.fc2 = LoRALinear(hidden_dim, num_classes, rank=rank, alpha=16)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

model = LoRAClassifier(rank=4)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")


## Step 3: Train Only LoRA Parameters


In [ ]:
#| eval: false
from torch.utils.data import TensorDataset, DataLoader

def make_tabular_data(n=2500, input_dim=64, num_classes=3):
    X = torch.randn(n, input_dim)
    y = torch.randint(0, num_classes, (n,))
    return TensorDataset(X, y)

ds = make_tabular_data()
train_ds, val_ds = torch.utils.data.random_split(ds, [2000, 500])
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=128)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LoRAClassifier(rank=4).to(device)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for ep in range(1, 8):
    model.train()
    tr_correct = tr_total = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        tr_correct += (logits.argmax(-1) == yb).sum().item()
        tr_total += len(yb)

    model.eval()
    va_correct = va_total = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(-1)
            va_correct += (pred == yb).sum().item()
            va_total += len(yb)

    print(f"Epoch {ep}: train_acc={tr_correct/tr_total:.3f}, val_acc={va_correct/va_total:.3f}")


## Step 4: Merge LoRA Weights for Inference


In [ ]:
#| eval: false
def merge_lora_linear(layer: LoRALinear) -> nn.Linear:
    merged = nn.Linear(layer.weight.size(1), layer.weight.size(0), bias=True)
    delta = layer.B @ layer.A
    merged.weight.data = layer.weight.data + layer.scaling * delta
    merged.bias.data = layer.bias.data.clone()
    return merged

merged_fc1 = merge_lora_linear(model.fc1)
print("Merged layer weight shape:", merged_fc1.weight.shape)


## Exercises

1. Compare LoRA ranks {2, 4, 8, 16}. How does accuracy scale with rank?
2. Add dropout in LoRA layers and measure overfitting effects.
3. Replace ReLU with GELU and compare validation metrics.
4. Benchmark trainable parameter counts against a full `nn.Linear` baseline.
5. Extend this tutorial to a transformer attention projection (`W_q`, `W_k`, `W_v`).
